In [ ]:
!pip install langchain
!pip install langchain_text_splitters
!pip install langchain_google_genai
!pip install langchain_community

In [ ]:
!pip install "unstructured[all-docs]"


In [ ]:
import os

In [ ]:
from google.colab import userdata
op = userdata.get('RAG')

In [ ]:
os.environ["GOOGLE_API_KEY"]= op

In [ ]:
# documnet loader
from langchain_community.document_loaders import PyPDFLoader
doc = PyPDFLoader("/content/HR-Policies-Manuals.pdf")
document = doc.load()


In [ ]:
document

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2020-08-26T06:56:00+00:00', 'author': 'hr', 'moddate': '2020-08-26T06:56:00+00:00', 'source': '/content/HR-Policies-Manuals.pdf', 'total_pages': 24, 'page': 0, 'page_label': '1'}, page_content='SPIL Corporate HR Policies  \n \n \nSIRCA PAINTS INDIA LTD \nNEW DELHI  \n \n \n \n \nCORPORATE  \n  HUMAN RESOURCES \nPOLICIES & MANUALS'),
 Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2020-08-26T06:56:00+00:00', 'author': 'hr', 'moddate': '2020-08-26T06:56:00+00:00', 'source': '/content/HR-Policies-Manuals.pdf', 'total_pages': 24, 'page': 1, 'page_label': '2'}, page_content='SPIL Corporate HR Policies  \n \n \n \n \nSection 1: Introduction  \n \nThis handbook is the summary of the policies, procedures, guidance and benefits to the employees \nand organization. It is an introduction to our vision, mission, values, what you expect from

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# text-spliter chunking
rts = RecursiveCharacterTextSplitter(separators=["\n\n","\n"," ",],chunk_size = 10,chunk_overlap=10)


In [ ]:
chunks = rts.split_documents(document)

In [ ]:
chunks

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2020-08-26T06:56:00+00:00', 'author': 'hr', 'moddate': '2020-08-26T06:56:00+00:00', 'source': '/content/HR-Policies-Manuals.pdf', 'total_pages': 24, 'page': 0, 'page_label': '1'}, page_content='SPIL'),
 Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2020-08-26T06:56:00+00:00', 'author': 'hr', 'moddate': '2020-08-26T06:56:00+00:00', 'source': '/content/HR-Policies-Manuals.pdf', 'total_pages': 24, 'page': 0, 'page_label': '1'}, page_content=' Corporate'),
 Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2020-08-26T06:56:00+00:00', 'author': 'hr', 'moddate': '2020-08-26T06:56:00+00:00', 'source': '/content/HR-Policies-Manuals.pdf', 'total_pages': 24, 'page': 0, 'page_label': '1'}, page_content='HR'),
 Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Wor

In [ ]:
pip install chromadb

In [ ]:
# embedding and vectore store
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embed = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-1.0"
)


In [ ]:
from langchain_community.vectorstores import Chroma

vdb = Chroma.from_documents(
    documents=chunks,
    embedding=embed,
    persist_directory="db"
)

vdb.persist()   # Important for saving to disk


GoogleGenerativeAIError: Error embedding content (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key expired. Please renew the API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key expired. Please renew the API key.'}]}}

In [ ]:
retiver = vdb.as_retriever(search_kwargs={"k":3})

In [ ]:
retiver.invoke("hi how are you")

In [ ]:
def f1(data):
  l=[]
  for y in retiver.invoke("hi how are you"):
    l.append(y.page_content)
  return "\n\n".join(l)

In [ ]:
from langchain_core.runnables import RunnablePassthrough,RunnableLambda,RunnableParallel,RunnableSequence

In [ ]:
r2 = RunnableLambda(f1)

NameError: name 'f1' is not defined

In [ ]:
chain1 = RunnableSequence(retiver,r2).invoke(" hi how are you")

NameError: name 'retiver' is not defined

In [ ]:
chain2 = RunnableParallel({"context":chain1,"question":RunnablePassthrough()})

NameError: name 'chain1' is not defined

In [ ]:
from langchain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate,HumanMessagePromptTemplate,SystemMessagePromptTemplate


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI


In [ ]:
cpt=ChatPromptTemplate.from_messages([SystemMessagePromptTemplate.from_template("you are a helpful hr assistance who is having 4 years of experinece"),
                                 HumanMessagePromptTemplate.from_template("""Answer the question based on the below given context and if there is no context then please return i dont know :
                                 context:{context}
                                 question:{question}""")])

In [ ]:
model=ChatGoogleGenerativeAI(model="gemini-1.5-flash")
sto=StrOutputParser()



In [ ]:
chain3=RunnableSequence(cpt,model,sto)

In [ ]:
rag_pipeline=RunnableSequence(chain2,chain3)

In [ ]:
rag_pipeline.invoke("how many leave i can take in a month")

In [ ]:
rag_pipeline.invoke("notice period")